# Airbnb Pricing Engine (XGBoost + SHAP)
Predicts nightly price and shows which features add the most money.

**How to run:** Runtime ▸ Run all. By default it creates the demo dataset; to use a real dataset, upload your CSV in step 2.

## 1. Install libraries

In [ ]:
!pip install -q xgboost shap pandas scikit-learn joblib matplotlib

## 2. Dataset
Either the demo dataset is generated, or upload your own CSV (same columns).

In [ ]:
"""Generate a synthetic Inside-Airbnb-style listings dataset (10,000 listings).

Prices follow realistic rules (location, room type, size, amenities, reviews)
plus noise, so a model has real signal to learn and SHAP gives sensible answers.
"""
import numpy as np
import pandas as pd

NEIGHBOURHOODS = {  # name: (price multiplier, lat, lon)
    "Downtown": (1.45, 34.050, -118.250), "Beachfront": (1.60, 34.010, -118.495),
    "Hollywood": (1.25, 34.098, -118.327), "Midtown": (1.10, 34.045, -118.340),
    "Arts District": (1.20, 34.040, -118.230), "Suburb North": (0.85, 34.200, -118.450),
    "Suburb East": (0.80, 34.080, -118.100), "Suburb South": (0.78, 33.900, -118.300),
    "Hills": (1.55, 34.120, -118.400), "Old Town": (0.95, 34.060, -118.200),
    "Airport Area": (0.82, 33.945, -118.400), "University": (0.90, 34.070, -118.440),
}
AMENITIES = {"wifi": 0.95, "kitchen": 0.85, "air_conditioning": 0.6, "washer": 0.55,
             "free_parking": 0.45, "gym": 0.15, "pool": 0.12, "hot_tub": 0.08}


def generate(path="data/airbnb_listings.csv", n=10000, seed=42):
    rng = np.random.default_rng(seed)
    names = list(NEIGHBOURHOODS)
    nb = rng.choice(names, n, p=np.array([.12, .05, .10, .10, .07, .09, .08, .08, .05, .10, .06, .10]))
    room = rng.choice(["Entire home/apt", "Private room", "Shared room"], n, p=[.6, .35, .05])
    room_mult = pd.Series(room).map({"Entire home/apt": 1.0, "Private room": 0.55, "Shared room": 0.32}).values

    acc = np.where(room == "Entire home/apt", rng.integers(2, 11, n),
                   np.where(room == "Private room", rng.integers(1, 4, n), rng.integers(1, 3, n)))
    bedrooms = np.clip(np.round(acc / 2.2 + rng.normal(0, .5, n)), 0, 8).astype(int)
    bathrooms = np.clip(np.round(acc / 4 + rng.normal(0, .4, n) * 2) / 2, 0.5, 4)
    am = {k: (rng.random(n) < p).astype(int) for k, p in AMENITIES.items()}
    superhost = (rng.random(n) < .22).astype(int)
    n_reviews = rng.poisson(rng.gamma(2, 15, n))
    score = np.clip(rng.normal(4.7, .25, n) + .1 * superhost, 3.0, 5.0).round(2)
    score = np.where(n_reviews == 0, np.nan, score)
    min_nights = rng.choice([1, 2, 3, 5, 7, 30], n, p=[.4, .25, .15, .08, .05, .07])
    availability = rng.integers(0, 366, n)

    nb_mult = pd.Series(nb).map({k: v[0] for k, v in NEIGHBOURHOODS.items()}).values
    lat = pd.Series(nb).map({k: v[1] for k, v in NEIGHBOURHOODS.items()}).values + rng.normal(0, .01, n)
    lon = pd.Series(nb).map({k: v[2] for k, v in NEIGHBOURHOODS.items()}).values + rng.normal(0, .01, n)

    log_price = (np.log(60) + np.log(nb_mult) + np.log(room_mult)
                 + 0.085 * acc + 0.14 * bedrooms + 0.08 * bathrooms
                 + 0.30 * am["pool"] + 0.18 * am["hot_tub"] + 0.10 * am["air_conditioning"]
                 + 0.07 * am["gym"] + 0.05 * am["free_parking"] + 0.04 * am["kitchen"]
                 + 0.03 * am["washer"] + 0.06 * superhost
                 + 0.35 * (np.nan_to_num(score, nan=4.5) - 4.5)
                 - 0.10 * (min_nights >= 30) + 0.0002 * np.minimum(n_reviews, 100)
                 + rng.normal(0, 0.20, n))
    price = np.round(np.exp(log_price), 0)

    df = pd.DataFrame({"neighbourhood": nb, "room_type": room, "accommodates": acc,
                       "bedrooms": bedrooms, "bathrooms": bathrooms, **am,
                       "host_is_superhost": superhost, "number_of_reviews": n_reviews,
                       "review_score": score, "minimum_nights": min_nights,
                       "availability_365": availability, "latitude": lat.round(5),
                       "longitude": lon.round(5), "price": price})
    df.to_csv(path, index=False)
    return df




In [ ]:
import os
os.makedirs("data", exist_ok=True); os.makedirs("outputs", exist_ok=True)
CSV = "data/airbnb_listings.csv"

# --- OPTION A (default): generate the demo dataset ---
generate(CSV)

# --- OPTION B: upload your own CSV (uncomment) ---
# from google.colab import files
# up = files.upload()              # pick your CSV
# CSV = list(up.keys())[0]

import pandas as pd
pd.read_csv(CSV).head()

## 3. Training code

In [ ]:
"""Airbnb pricing engine: predict nightly price + explain what drives it.

Model : XGBoost (falls back to sklearn HistGradientBoosting if XGBoost is missing)
Explain: SHAP summary plot if `shap` is installed, else permutation importance
Run   : python train.py --csv data/airbnb_listings.csv
Output: outputs/model.joblib (used by app.py), outputs/metrics.json, feature importance
"""
import argparse
import json
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, f1_score, mean_absolute_error,
                             mean_squared_error, precision_score, r2_score,
                             recall_score)
from sklearn.model_selection import train_test_split

CATEGORICAL = ["neighbourhood", "room_type"]
AMENITY_COLS = ["wifi", "kitchen", "air_conditioning", "washer", "free_parking", "gym", "pool", "hot_tub"]


def make_regressor():
    try:
        from xgboost import XGBRegressor
        return XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=6, subsample=0.8,
                            colsample_bytree=0.8, random_state=42, n_jobs=-1), "XGBoost"
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingRegressor
        return HistGradientBoostingRegressor(max_iter=400, learning_rate=0.06,
                                             random_state=42), "sklearn HistGradientBoosting (fallback)"


def prepare(df):
    X = df.drop(columns=["price"]).copy()
    X["review_score"] = X["review_score"].fillna(X["review_score"].median())
    X = pd.get_dummies(X, columns=CATEGORICAL).astype(float)
    return X


def main(csv_path="data/airbnb_listings.csv", out_dir="outputs"):
    os.makedirs(out_dir, exist_ok=True)
    df = pd.read_csv(csv_path).dropna(subset=["price"])
    X, y = prepare(df), df["price"].values
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

    model, model_name = make_regressor()
    model.fit(Xtr, np.log1p(ytr))                       # log-price target
    pred = np.expm1(model.predict(Xte))

    # classification view: is the listing "premium" (above the median training price)?
    thr = float(np.median(ytr))
    yc, pc = yte > thr, pred > thr
    metrics = {
        "_model": model_name, "premium_threshold_price": thr,
        "MAE_dollars": round(mean_absolute_error(yte, pred), 2),
        "RMSE_dollars": round(float(np.sqrt(mean_squared_error(yte, pred))), 2),
        "R2": round(r2_score(yte, pred), 4),
        "MAPE_percent": round(float(np.mean(np.abs(yte - pred) / yte) * 100), 2),
        "premium_accuracy": round(accuracy_score(yc, pc), 4),
        "premium_precision": round(precision_score(yc, pc), 4),
        "premium_recall": round(recall_score(yc, pc), 4),
        "premium_f1": round(f1_score(yc, pc), 4),
        "train_rows": len(Xtr), "test_rows": len(Xte),
    }

    # ---- explainability: which features add the most money ----
    try:
        import shap
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        sv = shap.TreeExplainer(model).shap_values(Xte.iloc[:1000])
        shap.summary_plot(sv, Xte.iloc[:1000], show=False)
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, "shap_summary.png"), dpi=150); plt.close()
        mean_abs = np.abs(sv).mean(axis=0)
        imp = pd.Series(mean_abs, index=X.columns, name="mean_abs_shap_log_price")
    except Exception:
        r = permutation_importance(model, Xte.iloc[:1500], np.log1p(yte[:1500]), n_repeats=3, random_state=0)
        imp = pd.Series(r.importances_mean, index=X.columns, name="permutation_importance")
    imp.sort_values(ascending=False).to_csv(os.path.join(out_dir, "feature_importance.csv"))

    # ---- save everything the Streamlit app needs ----
    meta = {"columns": list(X.columns), "amenities": AMENITY_COLS,
            "neighbourhoods": sorted(df.neighbourhood.unique().tolist()),
            "room_types": sorted(df.room_type.unique().tolist()),
            "nb_latlon": df.groupby("neighbourhood")[["latitude", "longitude"]].mean().to_dict("index"),
            "review_score_median": float(df.review_score.median())}
    joblib.dump({"model": model, "meta": meta}, os.path.join(out_dir, "model.joblib"))
    with open(os.path.join(out_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    print(json.dumps(metrics, indent=2))
    print("\nTop 10 price drivers:\n", imp.sort_values(ascending=False).head(10).round(4).to_string())
    return metrics




## 4. Train + evaluate (prints accuracy / precision / recall / F1)

In [ ]:
results = main(CSV, 'outputs')

## 5. Extras

In [ ]:
from IPython.display import Image, display
display(Image('outputs/shap_summary.png'))
pd.read_csv('outputs/feature_importance.csv', index_col=0).head(15)

## Download your outputs

In [ ]:
from google.colab import files
import shutil
shutil.make_archive("outputs", "zip", "outputs")
files.download("outputs.zip")